In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [4]:
import joblib
import sklearn

In [5]:
import os
import pandas as pd

from src.preprocessing import split_data, get_column_groups, build_preprocessor, save_preprocessor

In [6]:
df = pd.read_csv("../data/raw/10-employee_performance.csv")

df.head()

,employee_id,age,gender,education,department,job_level,years_at_company,years_in_current_role,monthly_salary,overtime_hours_monthly,num_projects_completed,performance_rating,training_hours_yearly,work_life_balance_score,distance_from_home_km,num_companies_worked,job_satisfaction,relationship_with_manager,stock_option_level,attrition
0,1,38,Female,Master,IT,3,6.3,0.2,6072.54,11,7,2,39.0,5,4.4,3,3,5,3,0
1,2,32,Male,Bachelor,Finance,1,3.4,3.8,3058.51,15,6,3,57.1,4,4.5,2,4,4,2,0
2,3,24,Female,Bachelor,Engineering,1,0.8,2.9,9270.48,5,6,5,41.1,1,49.6,2,3,1,0,1
3,4,33,Male,Bachelor,Sales,5,0.2,1.6,7567.36,3,8,3,21.2,2,0.7,2,5,4,1,0
4,5,51,Male,Bachelor,Finance,3,3.8,7.1,5621.87,5,8,4,42.6,4,6.6,3,3,4,2,1


In [7]:
target = "Performance Rating"

In [9]:
df.columns

Index(['employee_id', 'age', 'gender', 'education', 'department', 'job_level',
       'years_at_company', 'years_in_current_role', 'monthly_salary',
       'overtime_hours_monthly', 'num_projects_completed',
       'performance_rating', 'training_hours_yearly',
       'work_life_balance_score', 'distance_from_home_km',
       'num_companies_worked', 'job_satisfaction', 'relationship_with_manager',
       'stock_option_level', 'attrition'],
      dtype='str')

In [12]:
df.columns.str.lower()

Index(['employee_id', 'age', 'gender', 'education', 'department', 'job_level',
       'years_at_company', 'years_in_current_role', 'monthly_salary',
       'overtime_hours_monthly', 'num_projects_completed',
       'performance_rating', 'training_hours_yearly',
       'work_life_balance_score', 'distance_from_home_km',
       'num_companies_worked', 'job_satisfaction', 'relationship_with_manager',
       'stock_option_level', 'attrition'],
      dtype='str')

In [10]:
target = "performance_rating"

In [11]:
X_train, X_test, y_train, y_test = split_data(df, target)

In [13]:
X_train.shape, X_test.shape
y_train.value_counts()

performance_rating
3    4574
4    3353
2    1669
5    1035
1     569
Name: count, dtype: int64

In [14]:
num_cols, cat_cols = get_column_groups(X_train)

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

Numéricas: ['employee_id', 'age', 'job_level', 'years_at_company', 'years_in_current_role', 'monthly_salary', 'overtime_hours_monthly', 'num_projects_completed', 'training_hours_yearly', 'work_life_balance_score', 'distance_from_home_km', 'num_companies_worked', 'job_satisfaction', 'relationship_with_manager', 'stock_option_level', 'attrition']
Categóricas: ['gender', 'education', 'department']


In [15]:
preprocessor = build_preprocessor(num_cols, cat_cols)

In [16]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [17]:
X_train_processed.shape

(11200, 31)

In [18]:
save_preprocessor(preprocessor, "../data/processed/preprocessor.pkl")

In [19]:
import pandas as pd

X_train_df = pd.DataFrame(X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed)

X_train_df.head()

,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,-1.516759,0.393240,0.490751,-0.652176,0.522389,0.405670,0.357835,-0.802618,-0.534814,0.715336,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.103567,-0.215748,1.290579,-0.944830,-0.078210,-0.998118,1.066732,-2.431562,-1.452277,1.630130,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,-0.550253,1.437220,-0.309076,-0.652176,1.363227,-1.050657,-0.351062,-0.802618,-0.990197,0.715336,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.279602,1.350222,2.090407,-0.854782,0.001870,0.029454,0.003386,2.455269,-0.581692,-0.199458,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.522625,-1.520723,-1.108904,-0.337010,-0.638769,-1.145357,0.357835,-0.802618,0.329074,-0.199458,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [20]:
# columnas
num_cols, cat_cols = get_column_groups(X_train)

# preprocesador
preprocessor = build_preprocessor(num_cols, cat_cols)

# transformar
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# guardar
save_preprocessor(preprocessor, "../data/processed/preprocessor.pkl")

In [21]:
X_train_processed.shape

(11200, 31)

Se construyó un pipeline de preprocesamiento utilizando ColumnTransformer, aplicando imputación, escalado para variables numéricas y One-Hot Encoding para categóricas. Se aseguró consistencia entre train y test y se serializó el pipeline para reutilización en producción.

In [22]:
feature_names = preprocessor.get_feature_names_out()

X_all = df.drop(columns=[target])
X_all_processed = preprocessor.transform(X_all)

if hasattr(X_all_processed, "toarray"):
    X_all_processed = X_all_processed.toarray()

final_df = pd.DataFrame(X_all_processed, columns=feature_names)
final_df[target] = df[target].values

final_df.to_csv("../data/processed/final_dataset.csv", index=False)
final_df.head()

,num__employee_id,num__age,num__job_level,num__years_at_company,num__years_in_current_role,num__monthly_salary,num__overtime_hours_monthly,num__num_projects_completed,num__training_hours_yearly,num__work_life_balance_score,...,cat__education_PhD,cat__department_Engineering,cat__department_Finance,cat__department_HR,cat__department_IT,cat__department_Legal,cat__department_Marketing,cat__department_Operations,cat__department_Sales,performance_rating
0,-1.728267,-0.302746,0.490751,0.405880,-0.919049,0.350776,1.066732,0.419089,-0.052644,1.630130,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2
1,-1.728020,-0.824736,-1.108904,-0.246963,0.522389,-1.019776,2.484526,0.011853,1.159478,0.715336,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3
2,-1.727773,-1.520723,-1.108904,-0.832270,0.162029,1.804956,-1.059960,0.011853,0.087989,-2.029045,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
3,-1.727525,-0.737738,2.090407,-0.967341,-0.358490,1.030506,-1.768857,0.826325,-1.244676,-1.114252,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3
4,-1.727278,0.828232,0.490751,-0.156916,1.843706,0.145845,-1.059960,0.826325,0.188441,0.715336,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4


In [23]:
os.makedirs("../models", exist_ok=True)
save_preprocessor(preprocessor, "../models/preprocessing_pipeline.pkl")